In [9]:
from openai import OpenAI, AsyncOpenAI

from dotenv import load_dotenv
import os
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel
from openai.types.responses import ResponseTextDeltaEvent

import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content

load_dotenv(override=True)

True

In [10]:
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

In [11]:
@function_tool
def send_test_email(text: str)->str:
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("osiemomaina85@gmail.com")  # Change to your verified sender
    to_email = To("mainanorbert90@gmail.com")  # Change to your recipient
    content = Content("text/plain", text)
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

In [12]:
tools = [send_test_email]

In [13]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
)

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-4o-mini",          # or any other model on OpenRouter, e.g. "anthropic/claude-3.5-sonnet"
    messages=[{"role": "user", "content": "sende email with invite to join a meeting tommorow"}],
)

print(response.choices[0].message.content)

In [ ]:
llm_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
)
model = OpenAIChatCompletionsModel(
    model="openai/gpt-4o-mini",
    openai_client=llm_client,
)

In [ ]:
email_agent = Agent(
    name="Email Agent",
    instructions="You are an email agent that sends emails to the user, use send_test_email tool to send emails",
    tools =tools,
    model = model
    # openai_client = or_model
)

user_message = "send an email to the user and inform them meetingas started "

with trace("email agent") as runner:
    result = await Runner.run(email_agent, user_message)

print(result.final_output)



202
I have sent an email to inform the user that the meeting has started.
